# Loss functions

> A set of custom loss functions

In [ ]:
#| default_exp losses

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from bioMONAI.core import store_attr

In [ ]:
#| export
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import sigmoid

from monai.losses import SSIMLoss

from scipy.optimize import curve_fit

from bioMONAI.metrics import FRCMetric, get_fourier_ring_correlations


In [ ]:
show_doc(SSIMLoss)

---

### SSIMLoss

>      SSIMLoss (spatial_dims:int, data_range:float=1.0,
>                kernel_type:monai.metrics.regression.KernelType|str=gaussian,
>                win_size:int|collections.abc.Sequence[int]=11,
>                kernel_sigma:float|collections.abc.Sequence[float]=1.5,
>                k1:float=0.01, k2:float=0.03,
>                reduction:monai.utils.enums.LossReduction|str=mean)

Compute the loss function based on the Structural Similarity Index Measure (SSIM) Metric.

For more info, visit
    https://vicuesoft.com/glossary/term/ssim-ms-ssim/

SSIM reference paper:
    Wang, Zhou, et al. "Image quality assessment: from error visibility to structural
    similarity." IEEE transactions on image processing 13.4 (2004): 600-612.

### Combined Losses

In [ ]:
#| export

class CombinedLoss:
    "losses combined"
    def __init__(self, spatial_dims=2, alpha=0.33, beta=0.33):
        store_attr()
        self.SSIM_loss = SSIMLoss(spatial_dims=spatial_dims)
        self.MSE_loss =  nn.MSELoss()
        self.MAE_loss =  nn.L1Loss()
        
    def __call__(self, pred, targ):
        return (1 - self.alpha - self.beta) * self.SSIM_loss(pred, targ) + self.alpha * self.MSE_loss(pred, targ) + self.beta * self.MAE_loss(pred, targ)
        

In [ ]:
#| export
class MSSSIMLoss(nn.Module):
    def __init__(self, C1=0.01**2, C2=0.03**2, sigma=(0.5, 1., 2., 4., 8.)):
        super(MSSSIMLoss, self).__init__()
        self.C1 = C1
        self.C2 = C2
        self.sigma = sigma

    def forward(self, x, y):
        # Check dimensions
        if x.size() != y.size():
            raise ValueError("Input images must have the same dimensions.")
        if x.size(2) % 2 != 1 or y.size(2) % 2 != 1:
            raise ValueError("Odd patch size preferred")

        num_scale = len(self.sigma)
        width = x.size(2)
        batch_size = x.size(0)
        channels = x.size(1)

        # Initialize variables
        mux = torch.empty((num_scale, batch_size, channels, 1, 1), device=x.device)
        muy = torch.empty_like(mux)
        sigmax2 = torch.empty_like(mux)
        sigmay2 = torch.empty_like(mux)
        sigmaxy = torch.empty_like(mux)
        l = torch.empty_like(mux)
        cs = torch.empty_like(mux)
        Pcs = torch.ones((batch_size,), device=x.device)

        for i, sigma in enumerate(self.sigma):
            # Create Gaussian filter
            weights = torch.exp(-torch.arange(-(width // 2), width // 2 + 1, dtype=torch.float32, device=x.device)**2 / (2 * sigma**2))
            weights = torch.outer(weights, weights)
            weights = weights / torch.sum(weights)
            weights = weights.view(1, 1, width, width).expand(channels, 1, -1, -1)

            # Compute mean and variance
            mux[i] = F.conv2d(x, weights, padding=0, groups=channels)
            muy[i] = F.conv2d(y, weights, padding=0, groups=channels)
            sigmax2[i] = F.conv2d(x * x, weights, padding=0, groups=channels) - mux[i] ** 2
            sigmay2[i] = F.conv2d(y * y, weights, padding=0, groups=channels) - muy[i] ** 2
            sigmaxy[i] = F.conv2d(x * y, weights, padding=0, groups=channels) - mux[i] * muy[i]

            # Compute luminance and contrast-structure components
            l[i] = (2 * mux[i] * muy[i] + self.C1) / (mux[i] ** 2 + muy[i] ** 2 + self.C1)
            cs[i] = (2 * sigmaxy[i] + self.C2) / (sigmax2[i] + sigmay2[i] + self.C2)
            Pcs *= cs[i].view(batch_size, -1).prod(dim=-1)

        # Final MSSSIM calculation
        msssim = l[-1].view(batch_size, -1).prod(dim=-1) * Pcs
        loss = 1 - msssim.mean()

        return loss

    def backward(self, x, y):
        raise NotImplementedError("The backward function is not implemented as PyTorch handles gradients automatically.")


tensor(1.)


In [ ]:

msssim_loss = MSSSIMLoss()
output = torch.rand(10, 3, 33, 33)  # Example output
target = torch.rand(10, 3, 33, 33)  # Example target
loss = msssim_loss(output, target)
print(loss)


In [ ]:
#| export
class MSSSIML1Loss(nn.Module):
    def __init__(self, alpha=0.025, C1=0.01**2, C2=0.03**2, sigma=(0.5, 1., 2., 4., 8.)):
        super(MSSSIML1Loss, self).__init__()
        self.alpha = alpha
        self.msssim_loss = MSSSIMLoss(C1=C1, C2=C2, sigma=sigma)

    def forward(self, x, y):
        # Compute MSSSIM loss
        loss_MSSSIM = self.msssim_loss(x, y)

        # Compute L1 loss, weighted by a Gaussian filter
        l1_loss = F.l1_loss(x, y, reduction='none')
        gaussian_weight = self.get_gaussian_weight(x.size())
        weighted_l1_loss = (l1_loss * gaussian_weight).mean()

        # Combine the losses
        total_loss = self.alpha * loss_MSSSIM + (1 - self.alpha) * weighted_l1_loss
        return total_loss

    def get_gaussian_weight(self, size):
        """Generate a Gaussian weight tensor based on input size."""
        batch_size, channels, width, height = size
        sigma = width / 6.0  # Using width/6 as an approximate scale for sigma

        x = torch.arange(width, dtype=torch.float32, device='cuda')
        y = torch.arange(height, dtype=torch.float32, device='cuda')
        
        # Explicitly pass the indexing argument
        x_grid, y_grid = torch.meshgrid(x, y, indexing='ij')
        
        gaussian = torch.exp(-((x_grid - width//2)**2 + (y_grid - height//2)**2) / (2 * sigma**2))
        gaussian /= gaussian.sum()  # Normalize the Gaussian

        gaussian_weight = gaussian.view(1, 1, width, height)
        gaussian_weight = gaussian_weight.expand(batch_size, channels, -1, -1)
        return gaussian_weight


tensor(0.0253, device='cuda:0')


In [ ]:

msssim_l1_loss = MSSSIML1Loss()
output = torch.rand(10, 3, 33, 33).cuda()  # Example output
target = torch.rand(10, 3, 33, 33).cuda()  # Example target
loss = msssim_l1_loss(output, target)
print(loss)


In [ ]:
#| export
class MSSSIML2Loss(nn.Module):
    def __init__(self, alpha=0.1, C1=0.01**2, C2=0.03**2, sigma=(0.5, 1., 2., 4., 8.)):
        super(MSSSIML2Loss, self).__init__()
        self.alpha = alpha
        self.msssim_loss = MSSSIMLoss(C1=C1, C2=C2, sigma=sigma)

    def forward(self, x, y):
        # Compute MSSSIM loss
        loss_MSSSIM = self.msssim_loss(x, y)

        # Compute L2 loss, weighted by a Gaussian filter
        l2_loss = F.mse_loss(x, y, reduction='none')  # L2 loss (without reduction)
        gaussian_weight = self.get_gaussian_weight(x.size())
        weighted_l2_loss = (l2_loss * gaussian_weight).mean() / 2.0  # Weighted L2 loss

        # Combine the losses
        total_loss = self.alpha * loss_MSSSIM + (1 - self.alpha) * weighted_l2_loss
        return total_loss

    def get_gaussian_weight(self, size):
        """Generate a Gaussian weight tensor based on input size."""
        batch_size, channels, width, height = size
        sigma = width / 6.0  # Using width/6 as an approximate scale for sigma

        x = torch.arange(width, dtype=torch.float32, device='cuda')
        y = torch.arange(height, dtype=torch.float32, device='cuda')
        
        # Explicitly pass the indexing argument
        x_grid, y_grid = torch.meshgrid(x, y, indexing='ij')
        
        gaussian = torch.exp(-((x_grid - width//2)**2 + (y_grid - height//2)**2) / (2 * sigma**2))
        gaussian /= gaussian.sum()  # Normalize the Gaussian

        gaussian_weight = gaussian.view(1, 1, width, height)
        gaussian_weight = gaussian_weight.expand(batch_size, channels, -1, -1)
        return gaussian_weight


tensor(0.1001, device='cuda:0')


In [ ]:
msssim_l2_loss = MSSSIML2Loss()
output = torch.rand(10, 3, 33, 33).cuda()  # Example output
target = torch.rand(10, 3, 33, 33).cuda()  # Example target
loss = msssim_l2_loss(output, target)
print(loss)


### Dice Loss

In [ ]:
#| export

class DiceLoss(nn.Module):

    """
    DiceLoss computes the Sørensen–Dice coefficient loss, which is often used 
    for evaluating the performance of image segmentation algorithms.

    The Dice coefficient is a measure of overlap between two samples. It ranges 
    from 0 (no overlap) to 1 (perfect overlap). The Dice loss is computed as 
    1 - Dice coefficient, so it ranges from 1 (no overlap) to 0 (perfect overlap).

    Attributes:
        smooth (float): A smoothing factor to avoid division by zero and ensure numerical stability.

    Methods:
        forward(inputs, targets):
            Computes the Dice loss between the predicted probabilities (inputs) 
            and the ground truth (targets).
    """

    def __init__(self, smooth=1):

        """
        Initializes the DiceLoss instance with a smoothing factor.

        Args:
            smooth (float): A smoothing factor to avoid division by zero and ensure numerical stability.
                            Default is 1.
        """
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):
        
        # Make sure the inputs are probabilities
        inputs = sigmoid(inputs)

        # Flatten tensors
        inputs = inputs.view(-1)
        targets = targets.view(-1)

        # Calculate the intersection
        intersection = (inputs * targets).sum()

        # Compute Dice Coefficient
        dice = (2. * intersection + self.smooth) / (inputs.sum() + targets.sum() + self.smooth)

        # Copmute dice loss
        loss = 1 - dice

        return loss
        

In [ ]:
# inputs and targets must be equally dimensional tensors
from torch import randn, randint


Dice Loss: 0.4994280934333801


In [ ]:

inputs = randn((1, 1, 256, 256))  # Input
targets = randint(0, 2, (1, 1, 256, 256)).float()  # Ground Truth

# Initialize
dice_loss = DiceLoss()

# Compute loss
loss = dice_loss(inputs, targets)
print('Dice Loss:', loss.item())



### Fourier Ring Correlation

In [ ]:
#| export

def FRCLoss(image1, image2):

    """
    Compute the Fourier Ring Correlation (FRC) loss between two images.

    #### Args:
        - image1 (torch.Tensor): The first input image.
        - image2 (torch.Tensor): The second input image.

    #### Returns:
        - torch.Tensor: The FRC loss.
    """
    
    return (1 - FRCMetric(image1, image2))
    

In [ ]:
#| export
def seventh_fourier_ring_correlation(image1,image2):


    """
    Calculate the cutoff frequency at when Fourier ring correlation drops to 1/7.

    #### Args:
        - image1 (torch.Tensor): The first input image.
        - image2 (torch.Tensor): The second input image.

    #### Returns:
        - float: The cutoff frequency.
    """

    # Get y and x coordinates
    y, x = get_fourier_ring_correlations(image1, image2)

    # x -> frequency   y -> correlation
    x = x.numpy()
    y = y.numpy()


    # Exponential function to fit
    def exponential_func(x, a, b, c):
        return a * np.exp(-b * x) + c

    # Make fit
    params, _ = curve_fit(exponential_func, x, y, p0=[1, 1, 1])

    # Get Cutoff requency at 1/7
    cutoff_frequency = (exponential_func((1/7), *params))

    return cutoff_frequency

In [ ]:
show_doc(seventh_fourier_ring_correlation)

---

[source](https://github.com/bmandracchia/bioMONAI/blob/main/bioMONAI/losses.py#L241){target="_blank" style="float:right; font-size:smaller"}

### seventh_fourier_ring_correlation

>      seventh_fourier_ring_correlation (image1, image2)

Calculate the cutoff frequency at when Fourier ring correlation drops to 1/7.

#### Args:
    - image1 (torch.Tensor): The first input image.
    - image2 (torch.Tensor): The second input image.

#### Returns:
    - float: The cutoff frequency.

---

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()